In [60]:
import os
import subprocess
import textwrap
from pathlib import Path

from anthropic import Anthropic, omit
from anthropic.types import Message, MessageParam, TextBlock
from dotenv import dotenv_values

env_path = Path.cwd().parent / ".env.template"
for key, ref in dotenv_values(env_path).items():
    if ref is None:
        continue
    os.environ[key] = subprocess.run(
        ["op", "read", ref], capture_output=True, text=True, check=True
    ).stdout.strip()

client = Anthropic(
    base_url="https://openrouter.ai/api",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

models = {
    "deepseek": "deepseek/deepseek-v4.1-flash",
    "llama": "meta-llama/llama-3.1-8b-instruct",
    "mistral": "mistralai/mistral-small-3.1-24b-instruct",
    "haiku": "anthropic/claude-haiku-4.5",
    "qwen": "qwen/qwen-2.5-coder-32b-instruct",
}
model = models["deepseek"]

In [61]:
def get_reply(message: Message) -> str:
    text = next(
        (block.text for block in message.content if isinstance(block, TextBlock)),
        "",
    )
    if not text:
        return f"[no text block; stop_reason={message.stop_reason}, content={message.content!r}]"
    return text


def add_user_message(messages: list[MessageParam], text: str) -> None:
    user_message: MessageParam = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages: list[MessageParam], text: str) -> None:
    assistant_message: MessageParam = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def wrap_text(text: str, width: int = 120) -> str:
    return "\n".join(textwrap.fill(line, width=width) for line in text.splitlines())


# Start with an empty message list
messages: list[MessageParam] = []

In [76]:
def chat(
    messages: list[MessageParam],
    system: str | None = None,
    temperature: float = 1.0,
    stop_sequences: list[str] | None = None,
) -> str:
    message = client.messages.create(
        model=model,
        max_tokens=10000,
        thinking={"type": "disabled"},
        messages=messages,
        system=system or omit,
        temperature=temperature,
        stop_sequences=stop_sequences or omit,
    )
    return get_reply(message)

In [32]:
add_user_message(messages, "Create a 1 sentence fake database description.")
with client.messages.stream(
    model=model,
    max_tokens=10000,
    thinking={"type": "disabled"},
    messages=messages,
    temperature=0.999,
) as stream:
    for text in stream.text_stream:
        print(wrap_text(f"{text}"), end="")

print("\n---")
print(wrap_text(get_reply(stream.get_final_message())))

The Chrono-Spatial Empathy Index is a fictional database that catalogs and cross-references the emotional resonance of historical events across parallel timelines.
---
The Chrono-Spatial Empathy Index is a fictional database that catalogs and cross-references the emotional resonance of
historical events across parallel timelines.


In [ ]:
system = "You are a scientific assistant which popularizes complex scientific concepts."

while True:
    # You: Is it true that quantum computers consume hugh amount of energy? Why is it so?
    # You: how is physical qubit made? of what, silicone?
    user_input = input("You: ")
    if user_input == "/exit":
        break

    add_user_message(messages, user_input)
    print(wrap_text(f"You: {user_input}"))

    reply = chat(messages, system=system)
    add_assistant_message(messages, reply)

    print(wrap_text(f"AI: {reply}"))
    print("\n---\n")

You: Is it true that quantum computers consume hugh amount of energy? Why is it so?
AI: # Do Quantum Computers Really Use Huge Amounts of Energy?

**Short answer:** Compared to a laptop, yes — dramatically. Compared to the largest supercomputers, no — they actually
use far less. The "huge energy" impression comes from a real source, but it's rooted in *cooling and control*, not in
the computation itself.

---

## The Scale

- Many quantum computers use **superconducting qubits** that must be cooled to about **10–15 millikelvin** — colder than
outer space (~2.7 K).
- This requires a **dilution refrigerator**, which draws roughly **10–25 kW continuously**, just to keep a fingernail-
sized chip cold. That's the power draw of several homes.
- Add the room-temperature control electronics (microwave generators, cabling, racks): another **tens of kW**.
- **Total for a modest system: ~20–100 kW.**

For context, the Frontier supercomputer draws **~20,000 kW (20 MW)** — hundreds of times more th

In [84]:
model = models["llama"]

messages.clear()
add_user_message(messages, "Generate AWS EventBridge rule as JSON")
add_assistant_message(messages, "```json\n")
# Markdown(chat(messages, stop_sequences=["```"]).strip())
print(wrap_text(chat(messages, stop_sequences=["```"]).strip()))

{
  "name": "MyEventBridgeRule",
  "eventPattern": {
    "source": ["aws.s3"],
    "detail-type": ["S3 Object Created"],
    "detail": {
      "bucket": ["my-bucket"],
      "object-key": ["*.json"]
    }
  },
  "state_machine_arn": "arn:aws:states:us-east-1:123456789012:stateMachine:MyStateMachine",
  "tags": {
    "TagKey1": "TagValue1",
    "TagKey2": "TagValue2"
  },
  "roleArn": "arn:aws:iam::123456789012:role/MyRole"
}


In [59]:
messages.clear()
add_user_message(messages, "Generate 3 different short AWS CLI commands")
add_assistant_message(messages, "```bash")
print(chat(messages, stop_sequences=["```"]).strip())

<DSML>
# 1. List all S3 buckets in your account
aws s3 ls

# 2. Describe all running EC2 instances (IDs, type, state, IP)
aws ec2 describe-instances --filters "Name=instance-state-name,Values=running" --query "Reservations[].Instances[].{ID:InstanceId,Type:InstanceType,State:State.Name,IP:PublicIpAddress}" --output table

# 3. Get the identity of the caller making the request (useful for debugging credentials)
aws sts get-caller-identity
```
